# Day 2 Lab：让职业数字人开始有流程

在开始之前，我们先快速复习一下今天学了什么。

---

## 今天我们先讲清了一件事

Anthropic 把 AI 智能体系统分成两种：

- **workflow**
- **agent**

**workflow** 是你先把路设计好，AI 按路走。  
**agent** 是 AI 自己决定下一步要做什么。

这门课，我们先从 **workflow** 开始。

因为它更：

- 稳
- 清楚
- 可控

刚开始搭系统的时候，  
先把路搭稳，比一开始追求很高的自主性更重要。

---

## 今天我们还看了 5 种常见的 workflow pattern

### 1. Prompt Chaining｜提示链
一步接一步。  
前一步的结果，交给下一步。

### 2. Routing｜路由分流
先判断这是什么任务，  
再走不同的路。

### 3. Parallelization｜并行处理
几个分支同时做，  
最后再把结果合起来。

### 4. Evaluator-Optimizer｜评估—优化
先做一版，  
再检查一版，  
再继续优化。

### 5. Orchestrator-Workers｜总控—执行者
先看整个任务，  
再动态拆成几块，  
分给不同执行者去做。

---

## 今天最重要的感觉

这些模式不是拿来背的。  
它们更像是几种组织工作的方式。

真实系统里，  
往往会把几种模式组合起来用。

但流程也不是拆得越细越好。  
只有当后面的处理方式真的不同，  
这个分流才值得做。

---

## 那今天这份 notebook 要做什么？

今天，我们不加工具，  
也不加资料库。

我们先做一个最小的 **Routing system**。

也就是让职业数字人先学会：

**判断你现在要它干什么，  
再决定走哪条路。**

今天它先支持这几类任务：

- 自我介绍类
- 写作辅助类
- 数据分析类
- 工作安排 / 总结类
- 不明确问题先追问

---

## 今天这份 notebook 的目标

做完以后，你会得到：

**职业数字人 v1：会分流任务的助手**

它现在还不会真的分析表格，  
也还不会真的去读资料。

但它已经会先判断：

这个问题以后该往哪条路走。

---

好，接下来我们开始动手。

## 先做准备工作：把今天要用的大脑接上

在真正开始做 routing 之前，  
我们先把今天要用的大模型接上。

这里先做两件事：

- 导入今天会用到的工具
- 连接模型，准备一个最小调用函数

这样后面做任务分类和分流的时候，  
代码会更清楚，也更容易复用。

In [ ]:
# 导入工具
from openai import OpenAI
from dotenv import load_dotenv
import os
from IPython.display import display, Markdown

# 读取 .env 里的 API key
load_dotenv()

# 创建客户端（通义千问兼容 OpenAI 接口）
client = OpenAI(
    base_url="https://dashscope.aliyuncs.com/compatible-mode/v1",
    api_key=os.getenv("API_KEY")
)

def call_llm(user_question: str, system_prompt: str) -> str:
    response = client.chat.completions.create(
        model="qwen3.6-plus",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_question}
        ],
        temperature=0
    )
    return response.choices[0].message.content.strip()

display(Markdown("### ✅ 模型已连接"))

## 第一步：先把今天要分的几类任务说清楚

在开始写 routing 之前，  
我们先把今天的任务类别定下来。

因为一个最小的 routing system，  
最重要的第一步不是写很多代码，  
而是先知道：

**我们到底要分哪几类。**

今天，我们先支持这 5 类任务：

- `profile`：自我介绍类
- `writing`：写作辅助类
- `data_analysis`：数据分析类
- `work_planning`：工作安排 / 总结类
- `clarify`：不明确问题先追问

接下来，我们会先把这些类别定义出来。

In [ ]:
TASK_TYPES = {
    "profile": {
        "name_cn": "自我介绍类",
        "description": "用户想了解你是谁、你做什么、你做过什么项目。",
        "examples": [
            "你是谁？",
            "你是做什么的？",
            "你做过哪些项目？"
        ]
    },
    "writing": {
        "name_cn": "写作辅助类",
        "description": "用户希望你帮忙写、改、润色一段文字。",
        "examples": [
            "帮我写一段周报开头。",
            "帮我润色这段话。",
            "帮我写一个简短总结。"
        ]
    },
    "data_analysis": {
        "name_cn": "数据分析类",
        "description": "用户希望你分析表格、数据或结果。",
        "examples": [
            "帮我分析这份表格。",
            "看看这组数据有什么特点。",
            "总结一下这份表的数据重点。"
        ]
    },
    "work_planning": {
        "name_cn": "工作安排 / 总结类",
        "description": "用户希望你帮助整理待办、总结重点或安排工作。",
        "examples": [
            "我下周的工作重点是什么？",
            "帮我整理一下今天的待办。",
            "帮我总结一下最近的工作重点。"
        ]
    },
    "clarify": {
        "name_cn": "不明确问题先追问",
        "description": "用户的问题太模糊，先不要乱答，要先追问澄清。",
        "examples": [
            "帮我处理一下这个。",
            "你看怎么办？",
            "这个怎么弄？"
        ]
    }
}

print("今天的任务类别：\n")
for key, info in TASK_TYPES.items():
    print(f"{key} | {info['name_cn']}")
    print(f"说明：{info['description']}")
    print("示例：")
    for ex in info["examples"]:
        print(f" - {ex}")
    print()